# Champion V014 Production Training Notebook

This notebook reproduces the official Round 1 winning model:

**V014: CatBoost Alpha Pack Drop-Missing Seed Ensemble**

This model uses:
- train_dropmissing.csv and test_dropmissing.csv
- 35 Alpha Pack features
- Target-binned StratifiedKFold
- CatBoostRegressor
- Seeds: [42, 202, 777, 2026, 9999]
- 5 folds per seed
- Final prediction: average of seed predictions

This notebook also saves all model artifacts for the final-round MLOps system.

Imports

In [1]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

In [2]:
train_path = "../data/processed/train_dropmissing.csv"
test_path = "../data/processed/test_dropmissing.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

id_col = "record_id"
target = "flood_risk_score"

print("Train path:", train_path)
print("Test path:", test_path)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())
print("Duplicate train IDs:", train[id_col].duplicated().sum())
print("Duplicate test IDs:", test[id_col].duplicated().sum())
print("Train-only columns:", set(train.columns) - set(test.columns))
print("Test-only columns:", set(test.columns) - set(train.columns))
print("Target valid:", train[target].between(0, 1).all())

Train path: ../data/processed/train_dropmissing.csv
Test path: ../data/processed/test_dropmissing.csv
Train shape: (15518, 69)
Test shape: (5300, 68)
Train missing: 0
Test missing: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train-only columns: {'flood_risk_score'}
Test-only columns: set()
Target valid: True


Add Alpha Pack feature engineering

In [3]:
def add_alpha_pack_features(df):
    df = df.copy()
    eps = 1e-5
    
    if "distance_to_river_m_clipped" in df.columns:
        distance = df["distance_to_river_m_clipped"]
    else:
        distance = df["distance_to_river_m"].clip(lower=0)
    
    if "rainfall_7d_mm_clipped" in df.columns:
        rainfall_7d = df["rainfall_7d_mm_clipped"]
    else:
        rainfall_7d = df["rainfall_7d_mm"].clip(lower=0)
    
    inundation = df["inundation_area_sqm"].clip(lower=0)
    
    df["GOLDEN_distance_rainfall_ratio"] = (
        np.log1p(distance) - np.log1p(rainfall_7d + eps)
    )
    
    df["distance_to_river_DIV_inundation_area"] = (
        distance / (inundation + eps)
    )
    
    df["distance_to_river_DIV_rainfall_7d"] = (
        distance / (rainfall_7d + eps)
    )
    
    df["rainfall_7d_MULT_inundation_area"] = (
        rainfall_7d * inundation
    )
    
    new_cols = [
        "GOLDEN_distance_rainfall_ratio",
        "distance_to_river_DIV_inundation_area",
        "distance_to_river_DIV_rainfall_7d",
        "rainfall_7d_MULT_inundation_area"
    ]
    
    for col in new_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    return df

train_fe = add_alpha_pack_features(train)
test_fe = add_alpha_pack_features(test)

print("After feature engineering:")
print("Train shape:", train_fe.shape)
print("Test shape:", test_fe.shape)
print("Train missing:", train_fe.isnull().sum().sum())
print("Test missing:", test_fe.isnull().sum().sum())

After feature engineering:
Train shape: (15518, 73)
Test shape: (5300, 72)
Train missing: 0
Test missing: 0


Add exact 35 Alpha Pack feature list

In [4]:
alpha_pack_features = [
    "district",
    "distance_to_river_DIV_inundation_area",
    "distance_to_river_DIV_rainfall_7d",
    "GOLDEN_distance_rainfall_ratio",
    "generation_date",
    "reason_not_good_to_live",
    "inundation_area_sqm",
    "infrastructure_score",
    "extreme_weather_index",
    "terrain_roughness_index",
    "road_quality",
    "monthly_rainfall_mm_log1p",
    "landcover",
    "rainfall_7d_MULT_inundation_area",
    "place_name",
    "rainfall_7d_mm",
    "seasonal_index",
    "ndwi_qmap",
    "latitude",
    "longitude",
    "distance_to_river_m",
    "ndwi",
    "water_supply",
    "nearest_evac_km_log1p",
    "elevation_m_yeojohnson",
    "monthly_rainfall_mm",
    "socioeconomic_status_index",
    "population_density_per_km2_log1p",
    "water_presence_flag",
    "nearest_hospital_km_log1p",
    "ndvi_qmap",
    "flood_occurrence_current_event",
    "rainfall_7d_mm_log1p",
    "soil_type",
    "population_density_per_km2"
]

missing_train_features = [col for col in alpha_pack_features if col not in train_fe.columns]
missing_test_features = [col for col in alpha_pack_features if col not in test_fe.columns]

print("Number of Alpha Pack features:", len(alpha_pack_features))
print("Missing train features:", missing_train_features)
print("Missing test features:", missing_test_features)

if len(missing_train_features) > 0 or len(missing_test_features) > 0:
    raise ValueError("Some Alpha Pack features are missing. Stop and check columns.")

Number of Alpha Pack features: 35
Missing train features: []
Missing test features: []


Prepare Model Data

In [5]:
X = train_fe[alpha_pack_features].copy()
y = train_fe[target].copy()

X_test = test_fe[alpha_pack_features].copy()
test_ids = test_fe[id_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("Missing in X:", X.isnull().sum().sum())
print("Missing in X_test:", X_test.isnull().sum().sum())
print("Feature columns match:", list(X.columns) == list(X_test.columns))

cat_features = [
    col for col in X.columns
    if X[col].dtype == "object" or str(X[col].dtype) == "str"
]

print("Number of categorical features:", len(cat_features))
print(cat_features)

X shape: (15518, 35)
y shape: (15518,)
X_test shape: (5300, 35)
Missing in X: 0
Missing in X_test: 0
Feature columns match: True
Number of categorical features: 10
['district', 'generation_date', 'reason_not_good_to_live', 'road_quality', 'landcover', 'place_name', 'water_supply', 'water_presence_flag', 'flood_occurrence_current_event', 'soil_type']


Create target bins

In [6]:
target_bins = pd.cut(
    y,
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=[0, 1, 2, 3, 4]
)

print("Target bin distribution:")
print(target_bins.value_counts().sort_index())

Target bin distribution:
flood_risk_score
0    1899
1    3455
2    5993
3    2655
4    1516
Name: count, dtype: int64


Add submission validation function


In [7]:
def validate_submission(submission):
    print("Submission shape:", submission.shape)
    print(submission.head())
    
    print("Missing values:")
    print(submission.isnull().sum())
    
    print("Prediction range:")
    print("Min:", submission["flood_risk_score"].min())
    print("Max:", submission["flood_risk_score"].max())
    
    assert submission.shape[0] == len(test_ids)
    assert list(submission.columns) == ["record_id", "flood_risk_score"]
    assert submission["flood_risk_score"].isnull().sum() == 0
    assert submission["flood_risk_score"].between(0, 1).all()
    
    print("Submission is valid.")

Train V014 seed ensemble and save 25 models

In [8]:
model_dir = "../models/champion_v014_seed_ensemble"
os.makedirs(model_dir, exist_ok=True)

seeds = [42, 202, 777, 2026, 9999]
n_splits = 5

all_seed_oof_preds = []
all_seed_test_preds = []
seed_score_rows = []

for seed in seeds:
    print("\n" + "#" * 80)
    print(f"Training seed: {seed}")
    print("#" * 80)
    
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )
    
    seed_oof_preds = np.zeros(len(X))
    seed_test_preds = np.zeros(len(X_test))
    
    fold_rmse_scores = []
    fold_mae_scores = []
    fold_r2_scores = []
    
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, target_bins), 1):
        print("\n" + "=" * 70)
        print(f"Seed {seed} | Fold {fold}")
        
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        
        model = CatBoostRegressor(
            iterations=2000,
            learning_rate=0.03,
            depth=6,
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=seed + fold,
            verbose=300,
            early_stopping_rounds=200,
            allow_writing_files=False
        )
        
        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
            eval_set=(X_valid, y_valid),
            use_best_model=True
        )
        
        # Save each fold model for production ensemble
        model_path = f"{model_dir}/model_seed{seed}_fold{fold}.cbm"
        model.save_model(model_path)
        print("Saved model:", model_path)
        
        valid_preds = model.predict(X_valid)
        valid_preds = np.clip(valid_preds, 0, 1)
        
        seed_oof_preds[valid_idx] = valid_preds
        
        rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
        mae = mean_absolute_error(y_valid, valid_preds)
        r2 = r2_score(y_valid, valid_preds)
        
        fold_rmse_scores.append(rmse)
        fold_mae_scores.append(mae)
        fold_r2_scores.append(r2)
        
        print("Fold RMSE:", rmse)
        print("Fold MAE :", mae)
        print("Fold R2  :", r2)
        
        fold_test_preds = model.predict(X_test)
        seed_test_preds += fold_test_preds / n_splits
    
    seed_oof_preds = np.clip(seed_oof_preds, 0, 1)
    seed_test_preds = np.clip(seed_test_preds, 0, 1)
    
    seed_rmse = np.sqrt(mean_squared_error(y, seed_oof_preds))
    seed_mae = mean_absolute_error(y, seed_oof_preds)
    seed_r2 = r2_score(y, seed_oof_preds)
    
    print("\nSeed summary:")
    print("Seed:", seed)
    print("OOF RMSE:", seed_rmse)
    print("OOF MAE :", seed_mae)
    print("OOF R2  :", seed_r2)
    print("Test pred std:", seed_test_preds.std())
    
    seed_score_rows.append({
        "seed": seed,
        "oof_rmse": seed_rmse,
        "oof_mae": seed_mae,
        "oof_r2": seed_r2,
        "test_pred_min": seed_test_preds.min(),
        "test_pred_max": seed_test_preds.max(),
        "test_pred_mean": seed_test_preds.mean(),
        "test_pred_std": seed_test_preds.std()
    })
    
    all_seed_oof_preds.append(seed_oof_preds)
    all_seed_test_preds.append(seed_test_preds)


################################################################################
Training seed: 42
################################################################################

Seed 42 | Fold 1
0:	learn: 0.2364969	test: 0.2360924	best: 0.2360924 (0)	total: 71.3ms	remaining: 2m 22s
300:	learn: 0.2255166	test: 0.2317478	best: 0.2317092 (271)	total: 2.43s	remaining: 13.7s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.2316819405
bestIteration = 351

Shrink model to first 352 iterations.
Saved model: ../models/champion_v014_seed_ensemble/model_seed42_fold1.cbm
Fold RMSE: 0.23168194068370526
Fold MAE : 0.17632492961934465
Fold R2  : 0.038713533847676174

Seed 42 | Fold 2
0:	learn: 0.2367391	test: 0.2350707	best: 0.2350707 (0)	total: 33.2ms	remaining: 1m 6s
300:	learn: 0.2254899	test: 0.2303922	best: 0.2303872 (299)	total: 5.4s	remaining: 30.5s
600:	learn: 0.2174353	test: 0.2303459	best: 0.2302005 (420)	total: 10.6s	remaining: 24.6s
Stopped by overfitting detector 

Average predictions across seeds

In [9]:
ensemble_oof_preds = np.mean(all_seed_oof_preds, axis=0)
ensemble_test_preds = np.mean(all_seed_test_preds, axis=0)

ensemble_oof_preds = np.clip(ensemble_oof_preds, 0, 1)
ensemble_test_preds = np.clip(ensemble_test_preds, 0, 1)

ensemble_rmse = np.sqrt(mean_squared_error(y, ensemble_oof_preds))
ensemble_mae = mean_absolute_error(y, ensemble_oof_preds)
ensemble_r2 = r2_score(y, ensemble_oof_preds)

print("V014 Seed Ensemble Results")
print("Ensemble OOF RMSE:", ensemble_rmse)
print("Ensemble OOF MAE :", ensemble_mae)
print("Ensemble OOF R2  :", ensemble_r2)

print("Prediction distribution:")
print("Prediction min:", ensemble_test_preds.min())
print("Prediction max:", ensemble_test_preds.max())
print("Prediction mean:", ensemble_test_preds.mean())
print("Prediction std:", ensemble_test_preds.std())

V014 Seed Ensemble Results
Ensemble OOF RMSE: 0.23157158780580725
Ensemble OOF MAE : 0.17641904594511257
Ensemble OOF R2  : 0.04219244315562676
Prediction distribution:
Prediction min: 0.3642316667609831
Prediction max: 0.6413987232314529
Prediction mean: 0.4787833620514453
Prediction std: 0.04458173599051368


Create V014 submission

In [10]:
sub_v014 = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": ensemble_test_preds
})

validate_submission(sub_v014)

os.makedirs("../submissions", exist_ok=True)

sub_v014.to_csv(
    "../submissions/sub_v014_catboost_alpha_pack_dropmissing_seed_ensemble.csv",
    index=False
)

print("Saved submission.")

Submission shape: (5300, 2)
  record_id  flood_risk_score
0   F104559          0.466573
1   F100765          0.447645
2   F107573          0.472267
3   F110345          0.556143
4   F118850          0.517290
Missing values:
record_id           0
flood_risk_score    0
dtype: int64
Prediction range:
Min: 0.3642316667609831
Max: 0.6413987232314529
Submission is valid.
Saved submission.


Save reports and metadata

In [11]:
seed_scores_df = pd.DataFrame(seed_score_rows)

comparison = pd.DataFrame([
    {
        "version": "sub_v010_catboost_alpha_pack_dropmissing",
        "model": "Single CatBoost Alpha Pack Drop-Missing",
        "rows": 15518,
        "features": "35 Alpha Pack",
        "cv_rmse": 0.23180475883494558,
        "cv_mae": 0.17660570642311946,
        "cv_r2": 0.04023286825885215,
        "prediction_std": 0.044751843087494254,
        "public_score": 0.38176
    },
    {
        "version": "sub_v014_catboost_alpha_pack_dropmissing_seed_ensemble",
        "model": f"Seed Ensemble CatBoost, seeds={seeds}",
        "rows": len(train),
        "features": "35 Alpha Pack",
        "cv_rmse": ensemble_rmse,
        "cv_mae": ensemble_mae,
        "cv_r2": ensemble_r2,
        "prediction_std": ensemble_test_preds.std(),
        "public_score": None
    }
])

comparison["rmse_vs_v010"] = comparison["cv_rmse"] - comparison.loc[0, "cv_rmse"]
comparison["mae_vs_v010"] = comparison["cv_mae"] - comparison.loc[0, "cv_mae"]
comparison["r2_vs_v010"] = comparison["cv_r2"] - comparison.loc[0, "cv_r2"]
comparison["pred_std_vs_v010"] = comparison["prediction_std"] - comparison.loc[0, "prediction_std"]

os.makedirs("../reports", exist_ok=True)

seed_scores_df.to_csv("../reports/v014_seed_scores.csv", index=False)
comparison.to_csv("../reports/v014_seed_ensemble_comparison.csv", index=False)

# Also save inside model folder
seed_scores_df.to_csv(f"{model_dir}/seed_scores.csv", index=False)
comparison.to_csv(f"{model_dir}/model_comparison.csv", index=False)

with open(f"{model_dir}/feature_list.json", "w") as f:
    json.dump(alpha_pack_features, f, indent=4)

with open(f"{model_dir}/categorical_features.json", "w") as f:
    json.dump(cat_features, f, indent=4)

with open(f"{model_dir}/seeds.json", "w") as f:
    json.dump(seeds, f, indent=4)

model_metadata = {
    "model_name": "CatBoost Alpha Pack Drop-Missing Seed Ensemble",
    "model_version": "champion_v014_seed_ensemble",
    "model_type": "CatBoostRegressor Ensemble",
    "target": target,
    "id_column": id_col,
    "training_data": "train_dropmissing.csv",
    "test_data": "test_dropmissing.csv",
    "feature_count": len(alpha_pack_features),
    "categorical_feature_count": len(cat_features),
    "validation_method": "Target-binned StratifiedKFold",
    "n_splits": n_splits,
    "seeds": seeds,
    "number_of_models": len(seeds) * n_splits,
    "cv_rmse": float(ensemble_rmse),
    "cv_mae": float(ensemble_mae),
    "cv_r2": float(ensemble_r2),
    "prediction_min": float(ensemble_test_preds.min()),
    "prediction_max": float(ensemble_test_preds.max()),
    "prediction_mean": float(ensemble_test_preds.mean()),
    "prediction_std": float(ensemble_test_preds.std()),
    "round1_reference_public_score": 0.38176,
    "status": "Production Champion",
    "notes": "Official V014 Round 1 winning model: CatBoost Alpha Pack Drop-Missing Seed Ensemble."
}

with open(f"{model_dir}/model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=4)

print("Saved V014 reports and production artifacts.")
display(seed_scores_df)
display(comparison)

Saved V014 reports and production artifacts.


,seed,oof_rmse,oof_mae,oof_r2,test_pred_min,test_pred_max,test_pred_mean,test_pred_std
0,42,0.231808,0.176606,0.040235,0.367827,0.632363,0.479113,0.044752
1,202,0.231751,0.176810,0.040710,0.364201,0.642449,0.480106,0.044211
2,777,0.231717,0.176543,0.040992,0.362635,0.642047,0.477864,0.045169
3,2026,0.231919,0.176808,0.039317,0.361560,0.640691,0.478067,0.044315
4,9999,0.231840,0.176768,0.039971,0.364934,0.649443,0.478767,0.045063


,version,model,rows,features,cv_rmse,cv_mae,cv_r2,prediction_std,public_score,rmse_vs_v010,mae_vs_v010,r2_vs_v010,pred_std_vs_v010
0,sub_v010_catboost_alpha_pack_dropmissing,Single CatBoost Alpha Pack Drop-Missing,15518,35 Alpha Pack,0.231805,0.176606,0.040233,0.044752,0.38176,0.000000,0.000000,0.00000,0.00000
1,sub_v014_catboost_alpha_pack_dropmissing_seed_...,"Seed Ensemble CatBoost, seeds=[42, 202, 777, 2...",15518,35 Alpha Pack,0.231572,0.176419,0.042192,0.044582,NaN,-0.000233,-0.000187,0.00196,-0.00017


Final verification cell

In [12]:
saved_models = [
    f for f in os.listdir(model_dir)
    if f.endswith(".cbm")
]

print("Saved CatBoost models:", len(saved_models))
print("Expected models:", len(seeds) * n_splits)

assert len(saved_models) == len(seeds) * n_splits
assert os.path.exists(f"{model_dir}/feature_list.json")
assert os.path.exists(f"{model_dir}/categorical_features.json")
assert os.path.exists(f"{model_dir}/seeds.json")
assert os.path.exists(f"{model_dir}/model_metadata.json")
assert os.path.exists("../submissions/sub_v014_catboost_alpha_pack_dropmissing_seed_ensemble.csv")

print("V014 production model verification passed.")

Saved CatBoost models: 25
Expected models: 25
V014 production model verification passed.
